In [ ]:
"""
Análisis Tecno-Económico del Agregador en España
Extracción y Normalización de Precios del Mercado Diario (DAM)

Este script implementa un proceso ETL para depurar las series temporales de 
precios marginales casados en el Mercado Diario (OMIE), descargados vía ESIOS.
Incluye rutinas de limpieza para la eliminación de duplicados horarios y la 
alineación del vector de precios con el horizonte de optimización del modelo (24h).

Autor: Alberto Zafra Muñoz
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

# ==========================================
# 1. CONSTANTES Y CONFIGURACIÓN GENERAL
# ==========================================
ARCHIVO_ENTRADA = "Precio_DAM_3_4_2026.csv"
ARCHIVO_SALIDA = "Precios_DAM_Limpios.csv"
FECHA_ESTUDIO = '2026-04-03'

# ==========================================
# 2. PROCESAMIENTO DE DATOS (ETL)
# ==========================================
def procesar_precios_dam(ruta_csv: str, fecha_objetivo: str) -> Optional[pd.DataFrame]:
    """
    Lee el archivo bruto de ESIOS, filtra el horizonte de estudio, elimina 
    posibles lecturas duplicadas y formatea la serie de precios.

    Parámetros:
        ruta_csv (str): Ruta al archivo CSV con los datos del mercado.
        fecha_objetivo (str): Fecha de extracción en formato 'YYYY-MM-DD'.

    Retorna:
        pd.DataFrame: DataFrame con los precios marginales horarios limpios.
                      Devuelve None en caso de error de lectura.
    """
    print(f"[*] Iniciando extracción de datos de mercado desde: {ruta_csv}")
    
    if not os.path.exists(ruta_csv):
        print(f"[ERROR] No se encuentra el dataset fuente '{ruta_csv}'.")
        return None

    try:
        # Lectura del formato estándar de ESIOS (separador ';')
        df = pd.read_csv(ruta_csv, sep=";")
        
        # Conversión del vector temporal a objetos datetime nativos
        df['datetime'] = pd.to_datetime(df['datetime'])
        
        # Aislamiento de la jornada de simulación
        fecha_filtro = pd.to_datetime(fecha_objetivo).date()
        df_dia = df[df['datetime'].dt.date == fecha_filtro].copy()

        if df_dia.empty:
            print(f"[ERROR] No existen cotizaciones para la fecha {fecha_objetivo}.")
            return None

        # Transformación a índice horario (0-23)
        df_dia['Hora'] = df_dia['datetime'].dt.hour
        
        # Purga de datos: Selección de columnas, renombrado y eliminación de fallos de API (duplicados)
        df_clean = df_dia[['Hora', 'value']].rename(columns={'value': 'Precio_DAM_EUR_MWh'})
        df_clean = df_clean.drop_duplicates(subset=['Hora']).reset_index(drop=True)

        print(f"[*] Datos extraídos y purgados: {len(df_clean)} bloques horarios únicos consolidados.")
        return df_clean

    except Exception as e:
        print(f"[ERROR INESPERADO] Fallo durante la depuración de la serie de precios: {e}")
        return None

# ==========================================
# 3. REPRESENTACIÓN GRÁFICA ACADÉMICA
# ==========================================
def visualizar_precios_dam(df: pd.DataFrame, fecha: str):
    """
    Genera la figura con la evolución de los precios marginales del mercado SPOT.
    """
    plt.figure(figsize=(10, 5))
    plt.style.use('seaborn-v0_8-whitegrid')

    # Trazado de la curva de precios marginales
    plt.plot(df['Hora'], df['Precio_DAM_EUR_MWh'], 
             color='#800080', marker='o', linewidth=2.5, markersize=6, 
             label='Precio Marginal de Casación (OMIE)')
    
    # Sombreado del área bajo la curva
    plt.fill_between(df['Hora'], df['Precio_DAM_EUR_MWh'], 
                     color='#800080', alpha=0.15)

    # Línea de referencia del cero (crítica para precios negativos en España)
    plt.axhline(0, color='black', linewidth=1.2, linestyle='-')

    # Formateo y Estilos de la Figura
    plt.title(f'Evolución Horaria de los Precios Marginales (DAM) - {fecha}', 
              fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Hora del día ($t$)', fontsize=12)
    plt.ylabel('Precio del Mercado Diario (€/MWh)', fontsize=12)
    
    # Configuración de ejes
    plt.xticks(range(0, 24))
    plt.xlim(0, 23)
    
    # Ajuste dinámico del eje Y para soportar precios negativos si los hubiera
    y_min = min(0, df['Precio_DAM_EUR_MWh'].min() * 1.2)
    y_max = df['Precio_DAM_EUR_MWh'].max() * 1.15
    plt.ylim(y_min, y_max)
    
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='upper left', frameon=True, shadow=True, fontsize=11)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. BLOQUE PRINCIPAL DE EJECUCIÓN
# ==========================================
if __name__ == "__main__":
    df_resultado = procesar_precios_dam(ARCHIVO_ENTRADA, FECHA_ESTUDIO)
    
    if df_resultado is not None:
        # Guardar dataset limpio como input del optimizador Pyomo
        df_resultado.to_csv(ARCHIVO_SALIDA, index=False)
        print(f"[*] Serie de precios exportada satisfactoriamente a '{ARCHIVO_SALIDA}'.\n")
        
        # Mostrar preview en consola
        print("==================================================================")
        print(" VISTA PREVIA: VECTOR DE PRECIOS DEL MERCADO DIARIO (€/MWh)")
        print("==================================================================")
        print(df_resultado.head(24).to_string(index=False))
        print("==================================================================\n")
        
        print("[*] Generando modelo visual...")
        visualizar_precios_dam(df_resultado, FECHA_ESTUDIO)